## Import Libraries

In [7]:
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

## Import Datasets

In [13]:
#Load Datasets
fake_df = pd.read_parquet("/Volumes/E$/CEIR/Clean Dumps/Fake/fake_2026-06.parquet") # Update File name as needed
genuine_df = pd.read_parquet("/Volumes/E$/CEIR/Clean Dumps/Genuine/genuine_2026-06.parquet") # Update File name as needed

clone_df = pd.read_parquet("/Volumes/E$/CEIR/Clean Dumps/Cloned/all_cloned.parquet")

In [19]:
fake_df.head(2)

,imei_first_seen,last_seen,imei,imei_status,imsi,msisdn,rat,cgi,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-06-04 12:33:57,2026-05-30 09:21:13,00400300300300,W,641010407850068,256705990576,1,641-01-5030-16613,NATIONAL_ID,CF95010102J3CG,070,AIRTEL,Female,1995,31,KABAROLE,641,Uganda
1,2026-06-07 00:40:28,2026-06-01 16:01:19,05877465326124,W,641010420304827,256758979763,6,641-01-4710-24432,NATIONAL_ID,CF97105103WM1F,075,AIRTEL,Female,1997,29,LWENGO,641,Uganda


In [18]:
genuine_df.head(2)

,imei_first_seen,last_seen,tac,imei,imei_status,imsi,msisdn,rat,cgi,oem,...,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-06-09 22:05:17,2026-06-01 18:21:19,35523485,35523485148439,W,641010406336642,256746044262,1,641-01-9004-20912,Samsung Korea,...,NATIONAL_ID,CM01119100GA6J,074,AIRTEL,Male,2001,25,KYOTERA,641,Uganda
1,2026-06-06 07:35:53,2026-05-31 20:46:04,35523485,35523485153286,W,641010246053692,,6,641-10-1234-5678,Samsung Korea,...,NaN,NaN,NaN,AIRTEL,NaN,<NA>,<NA>,NaN,641,Uganda


In [17]:
clone_df.head(2)

,imei_first_seen,last_seen,tac,imei,imsi_count,msisdn_count,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,2026-07-01 21:02:08,2026-07-01 21:43:52,35804817,35804817426274,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-07-01 17:11:48,2026-07-01 21:20:53,35918236,35918236533982,2,2,Samsung Korea,Samsung,SM-A065F/DS,Galaxy A06,Smartphone,Android,14,2.0,1.0,1.0,1.0,0.0,2024.0


## Fake IMEI - Analysis

In [20]:

# FEATURE PREP — derive time parts + age brackets from raw data
# ============================================================
def prep_time_and_age(fake_df, time_col='imei_first_seen'):
    # Ensure the timestamp column is real datetime before using .dt
    fake_df[time_col] = pd.to_datetime(fake_df[time_col], errors='coerce')

    # Time dimensions used across the monthly reports
    fake_df['year_month'] = fake_df[time_col].dt.to_period('M')
    fake_df['hour']       = fake_df[time_col].dt.hour
    fake_df['dow']        = fake_df[time_col].dt.day_name()

    # Age -> ordered bracket. right=True means each bin is (low, high];
    # include_lowest=True so an exact age of 0 still lands in "<18".
    bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
    labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100-120"]
    numeric_age = pd.to_numeric(fake_df['age'], errors='coerce')
    age_bracket = pd.cut(numeric_age, bins=bins, labels=labels,
                         right=True, include_lowest=True)

    # Missing / out-of-range ages -> "Roamers" (no age data on file)
    age_bracket = age_bracket.astype('object').fillna('unknown')
    fake_df['age_bracket'] = pd.Categorical(age_bracket,
                                            categories=labels + ['unknown'],
                                            ordered=True)
    return fake_df


# REPORT HELPER — one month-by-dimension crosstab + display
# ============================================================
def report_by_month(df, col, title, fill=None, dropna=True, margins_name='TOTAL'):
    series = df[col]
    # Fill nulls when a label is given (cast to object first so it works
    # even if the column is categorical, e.g. mno/gender)
    if fill is not None:
        series = series.astype('object').fillna(fill)

    tab = pd.crosstab(df['year_month'], series, dropna=dropna,
                      margins=True, margins_name=margins_name)
    print(f"\n{title}\n{'='*70}")
    display(tab)
    return tab


# RUN — build features, then output the four monthly tables
# ============================================================
fake_df = prep_time_and_age(fake_df)

mno_tab      = report_by_month(fake_df, 'mno',         'Fake per MNO by month',       fill='unknown')
gender_tab   = report_by_month(fake_df, 'gender',      'Fake per gender by month',    fill='unknown')
age_tab      = report_by_month(fake_df, 'age_bracket', 'Fake per age group by month', dropna=False)
district_tab = report_by_month(fake_df, 'district',    'Fake per district by month')
country_tab  = report_by_month(fake_df, 'country',     'Fake per country by month',   fill='unknown')


Fake per MNO by month


mno,AIRTEL,MTN,unknown,TOTAL
year_month,,,,
2026-06,52913,10816,4095,67824
TOTAL,52913,10816,4095,67824



Fake per gender by month


gender,Female,Male,Undefined,unknown,TOTAL
year_month,,,,,
2026-06,21211,21918,2,24693,67824
TOTAL,21211,21918,2,24693,67824



Fake per age group by month


age_bracket,<18,18-30,31-40,41-50,51-60,61-70,71-100,100-120,unknown,TOTAL
year_month,,,,,,,,,,
2026-06,6,14363,13197,8266,4624,1871,803,0,24694,67824
TOTAL,6,14363,13197,8266,4624,1871,803,0,24694,67824



Fake per district by month


district,ABIM,ADJUMANI,AGAGO,ALEBTONG,AMOLATAR,AMUDAT,AMURIA,AMURU,APAC,ARUA,...,SHEEMA,SIRONKO,SOROTI,SSEMBABULE,TORORO,UNKNOWN,WAKISO,YUMBE,ZOMBO,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2026-06,29,36,60,75,40,6,135,52,57,255,...,407,407,177,553,852,151,1705,68,140,43131
TOTAL,29,36,60,75,40,6,135,52,57,255,...,407,407,177,553,852,151,1705,68,140,43131



Fake per country by month


country,Australia,Austria,Belgium,Burundi,Cambodia,Chad,China,Czech Republic,Democratic Republic of Congo,Egypt,...,South Korea,South Sudan,Sudan,Tanzania,Togo,Uganda,United Arab Emirates,United Kingdom,Zambia,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2026-06,3,4,1,1,1,1,34,1,1050,1,...,1,353,1,186,1,63729,30,9,1,67824
TOTAL,3,4,1,1,1,1,34,1,1050,1,...,1,353,1,186,1,63729,30,9,1,67824


## Genuine IMEI - Analysis

In [21]:

# FEATURE PREP — derive time parts + age brackets from raw data
# ============================================================
def prep_time_and_age(genuine_df, time_col='imei_first_seen'):
    # Ensure the timestamp column is real datetime before using .dt
    genuine_df[time_col] = pd.to_datetime(genuine_df[time_col], errors='coerce')

    # Time dimensions used across the monthly reports
    genuine_df['year_month'] = genuine_df[time_col].dt.to_period('M')
    genuine_df['hour']       = genuine_df[time_col].dt.hour
    genuine_df['dow']        = genuine_df[time_col].dt.day_name()

    # Age -> ordered bracket. right=True means each bin is (low, high];
    # include_lowest=True so an exact age of 0 still lands in "<18".
    bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
    labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100-120"]
    numeric_age = pd.to_numeric(genuine_df['age'], errors='coerce')
    age_bracket = pd.cut(numeric_age, bins=bins, labels=labels,
                         right=True, include_lowest=True)

    # Missing / out-of-range ages -> "Roamers" (no age data on file)
    age_bracket = age_bracket.astype('object').fillna('unknown')
    genuine_df['age_bracket'] = pd.Categorical(age_bracket,
                                                categories=labels + ['unknown'],
                                                ordered=True)
    return genuine_df


# REPORT HELPER — one month-by-dimension crosstab + display
# ============================================================
def report_by_month(df, col, title, fill=None, dropna=True, margins_name='TOTAL'):
    series = df[col]
    # Fill nulls when a label is given (cast to object first so it works
    # even if the column is categorical, e.g. mno/gender)
    if fill is not None:
        series = series.astype('object').fillna(fill)

    tab = pd.crosstab(df['year_month'], series, dropna=dropna,
                      margins=True, margins_name=margins_name)
    print(f"\n{title}\n{'='*70}")
    display(tab)
    return tab


# RUN — build features, then output the four monthly tables
# ============================================================
genuine_df = prep_time_and_age(genuine_df)

mno_tab      = report_by_month(genuine_df, 'mno',         'Genuine per MNO by month',       fill='unknown')
gender_tab   = report_by_month(genuine_df, 'gender',      'Genuine per gender by month',    fill='unknown')
age_tab      = report_by_month(genuine_df, 'age_bracket', 'Genuine per age group by month', dropna=False)
district_tab = report_by_month(genuine_df, 'district',    'Genuine per district by month')
country_tab  = report_by_month(genuine_df, 'country',     'Genuine per country by month',   fill='unknown')


Genuine per MNO by month


mno,AIRTEL,MTN,unknown,TOTAL
year_month,,,,
2026-06,607917,81982,103801,793700
TOTAL,607917,81982,103801,793700



Genuine per gender by month


gender,Female,Male,Undefined,unknown,TOTAL
year_month,,,,,
2026-06,253584,249159,4,290953,793700
TOTAL,253584,249159,4,290953,793700



Genuine per age group by month


age_bracket,<18,18-30,31-40,41-50,51-60,61-70,71-100,100-120,unknown,TOTAL
year_month,,,,,,,,,,
2026-06,34,150476,156242,101187,58974,24989,10843,0,290955,793700
TOTAL,34,150476,156242,101187,58974,24989,10843,0,290955,793700



Genuine per district by month


district,ABIM,ADJUMANI,AGAGO,ALEBTONG,AMOLATAR,AMUDAT,AMURIA,AMURU,APAC,ARUA,...,SHEEMA,SIRONKO,SOROTI,SSEMBABULE,TORORO,UNKNOWN,WAKISO,YUMBE,ZOMBO,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2026-06,319,509,698,896,692,240,1504,870,1216,3666,...,4883,5421,1923,6375,10300,1841,21602,986,2639,502747
TOTAL,319,509,698,896,692,240,1504,870,1216,3666,...,4883,5421,1923,6375,10300,1841,21602,986,2639,502747



Genuine per country by month


country,Algeria,Argentina,Australia,Austria,Bahrain,Belgium,Bosnia and Herzegovina,Botswana,Brazil,Burkina Faso,...,UNKNOWN,Uganda,Ukraine,United Arab Emirates,United Kingdom,Uruguay,Vietnam,Zambia,Zimbabwe,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2026-06,1,1,36,117,20,1593,1,4,3,2,...,18,689899,5,1467,1368,2,7,93,7,793700
TOTAL,1,1,36,117,20,1593,1,4,3,2,...,18,689899,5,1467,1368,2,7,93,7,793700
